# Chapter 6 — Matrix Decomposition: SVD and Compression

Companion notebook for *The Math That Powers AI* (2nd ed.), Chapter 6.

We work through the chapter's decompositions using the `mathpowersai.decompositions` package:

1. Eigendecomposition of a **symmetric** matrix (the Spectral Theorem: real eigenvalues, orthogonal eigenvectors)
2. The **SVD** $A = U \Sigma V^T$ and exact reconstruction
3. **Truncated SVD** and the Eckart–Young error formula $\|A - A_k\|_F = \sqrt{\sum_{i>k} \sigma_i^2}$
4. A **numerical Eckart–Young demonstration**: the truncated SVD beats random rank-$k$ factorizations
5. **Rank-$k$ storage savings**, including the chapter's $1000 \times 1000$ rank-10 checkpoint
6. **LU solve** and **Gram–Schmidt QR**

All randomness is seeded with `np.random.default_rng(42)` so every run is deterministic.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

In [ ]:
from mathpowersai.decompositions import (
    eigendecomposition,
    symmetric_eigendecomposition,
    svd,
    truncated_svd,
    low_rank_error,
    rank_k_storage,
    random_rank_k_factorization,
    lu_decomposition,
    lu_solve,
    gram_schmidt_qr,
)

rng = np.random.default_rng(42)

## 1. Eigendecomposition of a symmetric matrix

For a general square matrix, `eigendecomposition` solves $A v = \lambda v$ and the eigenvalues may be complex. But the **Spectral Theorem** says that if $A$ is symmetric ($A = A^T$):

- all eigenvalues are **real**,
- there are $n$ **orthonormal** eigenvectors, and
- $A = Q \Lambda Q^T$ with $Q$ orthogonal ($Q^T Q = I$).

`symmetric_eigendecomposition` uses `np.linalg.eigh`, which guarantees real output for symmetric input. We verify this on the chapter's *Try It* matrix $B = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$, whose eigenvalues are $\lambda = 1, 3$.

In [ ]:
B = np.array([[2.0, 1.0], [1.0, 2.0]])
evals, Q = symmetric_eigendecomposition(B)

print("B =\n", B)
print("eigenvalues (ascending):", np.round(evals, 6))   # book: 1, 3
print("eigenvalue dtype (real!):", evals.dtype)
print("Q^T Q = I:", np.allclose(Q.T @ Q, np.eye(2)))
print("Q Lambda Q^T == B:", np.allclose(Q @ np.diag(evals) @ Q.T, B))

# Contrast with the general (non-symmetric) path from Example 6.1:
A = np.array([[4.0, 1.0], [2.0, 3.0]])
w, V = eigendecomposition(A)
order = np.argsort(w.real)[::-1]
print("\nA = [[4, 1], [2, 3]] eigenvalues:", np.round(w.real[order], 6))  # book: 5, 2

## 2. SVD and reconstruction

Every matrix $A \in \mathbb{R}^{m \times n}$ — square or not, symmetric or not — factors as

$$A = U \Sigma V^T,$$

with $U$ and $V$ orthogonal and $\Sigma$ diagonal with non-negative singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$.

The chapter's worked example uses $A = \begin{bmatrix} 3 & 2 \\ 2 & 3 \\ 2 & -2 \end{bmatrix}$ with singular values $\sigma_1 = 5$, $\sigma_2 = 3$. We confirm the values, the orthonormality of the factors, and that multiplying the factors back together reconstructs $A$ to machine precision.

In [ ]:
M = np.array([[3.0, 2.0], [2.0, 3.0], [2.0, -2.0]])
U_m, S_m, Vt_m = svd(M)

print("singular values:", np.round(S_m, 6))  # book: 5, 3
print("U shape:", U_m.shape, " S shape:", S_m.shape, " Vt shape:", Vt_m.shape)
print("U^T U = I:", np.allclose(U_m.T @ U_m, np.eye(2)))
print("V V^T = I:", np.allclose(Vt_m.T @ Vt_m, np.eye(2)))

recon = U_m @ np.diag(S_m) @ Vt_m
print(f"reconstruction error ||A - U S V^T||_F: {np.linalg.norm(M - recon):.2e}")

# Sanity check on a symmetric 2x2 from the chapter trace: sigma = 4, 2
_, S_sym, _ = svd(np.array([[3.0, 1.0], [1.0, 3.0]]))
print("singular values of [[3, 1], [1, 3]]:", np.round(S_sym, 6))  # book: 4, 2

## 3. Truncated SVD and the Eckart–Young error formula

Keeping only the $k$ largest singular values gives the rank-$k$ approximation

$$A_k = \sum_{i=1}^{k} \sigma_i\, u_i v_i^T = U_k \Sigma_k V_k^T,$$

and the **Eckart–Young–Mirsky theorem** tells us its error *exactly*:

$$\|A - A_k\|_F = \sqrt{\sum_{i=k+1}^{r} \sigma_i^2}.$$

Below, the directly measured error `low_rank_error(G, k)` matches the square root of the sum of the **discarded** squared singular values for every $k$ — the discarded tail of the spectrum is the whole story.

In [ ]:
G = rng.standard_normal((12, 8))
_, sigma, _ = svd(G)
print("singular values of the seeded 12 x 8 matrix:")
print(np.round(sigma, 4))
print()

for k in (1, 2, 4, 6):
    measured = low_rank_error(G, k)
    predicted = float(np.sqrt(np.sum(sigma[k:] ** 2)))
    print(
        f"k = {k}: ||A - A_k||_F = {measured:.6f}, "
        f"sqrt(sum_(i>k) sigma_i^2) = {predicted:.6f}, "
        f"match = {np.isclose(measured, predicted)}"
    )

A_2 = truncated_svd(G, 2)
print("\nshape of A_2:", A_2.shape, " rank of A_2:", np.linalg.matrix_rank(A_2))

## 4. Eckart–Young numerically: SVD beats random rank-$k$ factorizations

Eckart–Young says $A_k$ is the **best possible** rank-$k$ approximation in the Frobenius norm — not just a good one. We can check this numerically: generate many seeded random rank-$k$ matrices $BC$ (with $B \in \mathbb{R}^{m \times k}$, $C \in \mathbb{R}^{k \times n}$) via `random_rank_k_factorization` and compare their approximation errors against the truncated SVD's.

No random factorization should ever do better — and across hundreds of trials, none does.

In [ ]:
m, n, k = 12, 8, 3
n_trials = 500

svd_err = low_rank_error(G, k)

random_errs = np.array([
    np.linalg.norm(G - random_rank_k_factorization(m, n, k, rng), "fro")
    for _ in range(n_trials)
])

print(f"truncated-SVD error at k = {k}:        {svd_err:.6f}")
print(f"best of {n_trials} random rank-{k} trials:  {random_errs.min():.6f}")
print(f"mean random error:                   {random_errs.mean():.6f}")
print(f"trials beating the SVD:              {int((random_errs < svd_err).sum())} / {n_trials}")

assert svd_err <= random_errs.min(), "Eckart-Young violated?!"
print("\nEckart-Young holds: no random rank-k factorization beat the truncated SVD.")

## 5. Rank-$k$ storage savings

Why compress at all? Storing the rank-$k$ factors $U_k$ ($mk$ values), $\Sigma_k$ ($k$ values), and $V_k$ ($nk$ values) costs

$$k\,(m + n + 1) \quad \text{values, versus} \quad mn \ \text{for the full matrix}.$$

The chapter's image-compression example: a $1000 \times 1000$ image at $k = 50$ takes $100{,}050$ values — about a 10:1 ratio.

**Chapter checkpoint:** a rank-10 factorization $A = UV^T$ of a $1000 \times 1000$ matrix stores $1000 \cdot 10 + 1000 \cdot 10 = 20{,}000$ elements — a **50x reduction** over the $1{,}000{,}000$ entries of the full matrix.

In [ ]:
m_img = n_img = 1000
full = m_img * n_img

print(f"full {m_img} x {n_img} matrix: {full:,} values\n")
for k_img in (10, 50, 100):
    compressed = rank_k_storage(m_img, n_img, k_img)
    print(
        f"k = {k_img:>3}: {compressed:>9,} values "
        f"(ratio {full / compressed:.1f}:1)"
    )

# Chapter checkpoint: rank-10 factorization A = U V^T (no separate Sigma)
rank10 = m_img * 10 + n_img * 10
print(
    f"\ncheckpoint -- rank-10 A = U V^T of 1000 x 1000: "
    f"{rank10:,} elements ({full // rank10}x reduction)"
)  # book: 20,000 elements, 50x reduction

## 6. LU solve and Gram–Schmidt QR

Two more workhorse factorizations from the chapter.

**LU (Algorithm 6.1).** Factor $A = LU$ once ($O(n^3)$), then solve $Ly = b$ by forward substitution and $Ux = y$ by backward substitution (each $O(n^2)$). We use the chapter's Example 6.4 matrix.

**QR via Gram–Schmidt (Algorithm 6.2).** Orthonormalize the columns of $A$ one at a time: subtract projections $R_{ij} = q_i^T a_j$ onto previous columns, normalize with $R_{jj} = \|v_j\|$. The result is $A = QR$ with $Q^T Q = I$ and $R$ upper triangular.

In [ ]:
# --- LU decomposition and solve (Example 6.4) ---
A_lu = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
L, U_lu = lu_decomposition(A_lu)
print("L =\n", L)
print("U =\n", U_lu)
print("L U == A:", np.allclose(L @ U_lu, A_lu))

b = np.array([1.0, 2.0, 3.0])
x = lu_solve(A_lu, b)
print("solve A x = [1, 2, 3]: x =", np.round(x, 6))
print(f"residual ||A x - b||: {np.linalg.norm(A_lu @ x - b):.2e}")

In [ ]:
# --- QR via Gram-Schmidt (Algorithm 6.2) ---
A_qr = rng.standard_normal((6, 4))
Q_qr, R = gram_schmidt_qr(A_qr)

print("Q shape:", Q_qr.shape, " R shape:", R.shape)
print("Q^T Q = I:", np.allclose(Q_qr.T @ Q_qr, np.eye(4)))
print("R upper triangular:", np.allclose(R, np.triu(R)))
print("Q R == A:", np.allclose(Q_qr @ R, A_qr))
print("\nR =\n", np.round(R, 4))

## Summary

- Symmetric matrices have **real** eigenvalues and an orthogonal eigenbasis: $A = Q \Lambda Q^T$ (Spectral Theorem).
- The SVD $A = U \Sigma V^T$ exists for *every* matrix and reconstructs it to machine precision.
- The truncated SVD's error is exactly $\sqrt{\sum_{i>k} \sigma_i^2}$, and by Eckart–Young no other rank-$k$ matrix — including hundreds of random factorizations — can do better.
- A rank-10 factorization of a $1000 \times 1000$ matrix needs only 20,000 numbers: a 50x reduction.
- LU turns repeated solves into cheap triangular substitutions; Gram–Schmidt QR builds an orthonormal basis column by column.